In [3]:
from IPython.display import display, HTML
display(HTML("""
<style>
div.container{width:85% !important;}
div.cell.code_cell.rendered{width:100%;}
div.input_prompt{padding:0px;}
div.CodeMirror {font-family:Consolas; font-size:20pt;}
div.output {font-size:12pt; font-weight:bold;}
div.input {font-family:Consolas; font-size:20pt;}
div.prompt {min-width:70px;}
div#toc-wrapper{padding-top:120px;}
div.text_cell_render ul li{font-size:12pt;padding:5px;}
table.dataframe{font-size:22px;}

</style>
"""))

**<font size="6" color="red">ch1허깅페이스 모델 사용</font>**
- Inference API 이용 : 모델의 결과를 server에서 
- pipline() 이용 : 모델을 다운받아 모델의 결과를 local에서 

- 허깅페이스 transformer에서 지원하는 task

|task값| 설명 |
|:--- | :--- |
|text-classification (별칭 sentiment-analysis)|	감정 분석, 뉴스 분류, 리뷰 분류 등 문장 분류|
|zero-shot-classification	|레이블에 대한 별도 학습 없이 후보 레이블 중에서 분류|
|text-generation	        |GPT 계열 모델을 이용한 텍스트 생성|
|fill-mask	                |문장 안의 빈칸(마스크)에 들어갈 단어 예측|
|ner (token-classification의 별칭)|	개체명 인식(사람, 조직, 장소 등 라벨링)|
|question-answering	|주어진 지문(context)을 근거로 질문에 답변|
|summarization	            |긴 문서를 짧게 요약|
|translation	            |서로 다른 언어 간 번역|
|image-to-text	            |이미지 내용을 설명하는 문장 생성|
|image-classification	    |이미지가 어떤 대상인지 분류|

 
- 처음 모델 사용시 "c:/Users/내컴퓨터이름/.cache/huggingface"에 다운로드 되느라 시간이 걸림    

In [2]:
import warnings
import os
import logging
 
# 경고 메시지 제거
warnings.filterwarnings('ignore')
 
# transformers 라이브러리의 로깅 레벨을 ERROR로 조정 (경고 숨김)
logging.getLogger("transformers").setLevel(logging.ERROR)
 
# Hugging Face 캐시 관련 symlink 경고 제거
os.environ['HF_HUB_DISABLE_SYMLINKS_WARNING'] = '1'
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'


## 1.텍스트 기반 감정분석(긍정/부정)
- 토큰화 -> 워드임배딩 -> 모델 -> predict : pipline()함수는 이 단계를 내부적으로 해 줌

In [5]:
from transformers import pipeline
classifier = pipeline(task="text-classification",
                     model="distilbert/distilbert-base-uncased-finetuned-sst-2-english")
    
classifier("I`ve been waiting for a Hugging face course my whole life.")

Device set to use cpu


In [8]:
# 특정 모델의 파라미터와 용량
from transformers import AutoModel
model=AutoModel.from_pretrained("distilbert/distilbert-base-uncased-finetuned-sst-2-english")
# 전체 파라미터 수 
total_params = sum(p.numel() for p in model.parameters())
print(f"전체 파라미터 수: {total_params:,}")
print(f"전체 파라미터 수: {total_params/1024/1024:.3f}MB")

전체 파라미터 수: 66,362,880
전체 파라미터 수: 63.289MB


In [9]:
from transformers import pipeline
classifier = pipeline(task = "text-classification",
                     model = "distilbert/distilbert-base-uncased-finetuned-sst-2-english")
# 감정분석할 내용이 많으면 list
classifier([
    "I`ve been waiting for a Hugging face course my whole life.",            
    "I hate this so much!"
])

Device set to use cpu


[{'label': 'POSITIVE', 'score': 0.9978979825973511},
 {'label': 'NEGATIVE', 'score': 0.9994558691978455}]

In [10]:
classifier("이 영화 재미있어요.")

[{'label': 'POSITIVE', 'score': 0.8855457305908203}]

In [11]:
classifier(["I love you","I hate you","힘들어요"])

[{'label': 'POSITIVE', 'score': 0.9998656511306763},
 {'label': 'NEGATIVE', 'score': 0.9991129040718079},
 {'label': 'POSITIVE', 'score': 0.8669537305831909}]

In [13]:
classifier = pipeline(task= "sentiment-analysis",
                     model = "daekeun-ml/koelectra-small-v3-nsmc")
texts = ['힘들어요','오늘기분최고야','당신이 싫어요','니가 참좋아']
classifier(texts)

Device set to use cpu


[{'label': '0', 'score': 0.9957089424133301},
 {'label': '1', 'score': 0.9991125464439392},
 {'label': '0', 'score': 0.793237030506134},
 {'label': '1', 'score': 0.9421136379241943}]

In [14]:
for text, result in zip(texts, classifier(texts)):
    label = "긍정" if result['label']=='1' else "부정"
    print(f"'{text}'->{label}{result['score']:.2%}")

'힘들어요'->부정99.57%
'오늘기분최고야'->긍정99.91%
'당신이 싫어요'->부정79.32%
'니가 참좋아'->긍정94.21%


## 2. 제로샷(zero-shot)

In [19]:
classifier = pipeline(task='zero-shot-classification'
                     model = 'facebook/bart-large-mnli')
classifier("I have a problem with my iphone that needs to be resolved asap!!",
          candidate_labels=["urgent", "not urgent", "phone", "tablet", "computer"])


No model was supplied, defaulted to facebook/bart-large-mnli and revision d7645e1 (https://huggingface.co/facebook/bart-large-mnli).
Using a pipeline without specifying a model name and revision in production is not recommended.
Device set to use cpu


In [ ]:
classifier("This is a course about the Transformers library",
            candidate_labels=["education", "phone", "business"])